# 🍄 Mushroom Classification with ViT

**Перед початком:** Runtime → Change runtime type → **GPU (T4)**

## 1. Налаштування середовища

In [ ]:
# Перевірка GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не знайдено! Перейдіть в Runtime → Change runtime type → GPU")

In [ ]:
# Підключення Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive підключено")

In [ ]:
# Клонування репозиторію
import os
os.chdir('/content')

if os.path.exists('Applied_Mashroomatics'):
    os.chdir('Applied_Mashroomatics')
    !git pull
else:
    !git clone https://github.com/konstanta-asya/Applied_Mashroomatics.git
    os.chdir('Applied_Mashroomatics')

print(f"✅ Робоча директорія: {os.getcwd()}")

In [ ]:
# Встановлення залежностей
!pip install -q tqdm

In [ ]:
# Підключення даних з Google Drive
import os

# Видаляємо стару папку якщо є
!rm -rf data/raw
!mkdir -p data/raw

# Створюємо симлінки на дані
!ln -s /content/drive/MyDrive/raw/DF20M data/raw/DF20M
!ln -s /content/drive/MyDrive/raw/DF20M-metadata data/raw/DF20M-metadata

# Перевірка
if os.path.exists('data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv'):
    print("✅ Дані підключено успішно")
    !ls data/raw/DF20M-metadata/
else:
    print("❌ Дані не знайдено! Перевірте шлях до папки raw на Google Drive")

## 2. Тренування моделі

**Варіанти:**
- 🚀 **Швидке** (~20-30 хв): `freeze_backbone=True`, 3-5 епох
- 🎯 **Повне** (~2-3 год): `freeze_backbone=False`, 10+ епох

In [ ]:
# ========== НАЛАШТУВАННЯ ==========

BATCH_SIZE = 32          # 32 для T4, можна 64 якщо вистачає пам'яті
EPOCHS = 5               # 3-5 для швидкого, 10-30 для повного
LEARNING_RATE = 3e-4     # 3e-4 для freeze, 1e-4 для повного
FREEZE_BACKBONE = True   # True = швидко, False = краща якість

# ==================================

freeze_flag = "--freeze_backbone" if FREEZE_BACKBONE else ""

!python src/train_vit.py \
    --csv_path data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv \
    --data_dir data/raw/DF20M \
    --batch_size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --lr {LEARNING_RATE} \
    {freeze_flag} \
    --checkpoint_dir /content/drive/MyDrive/mushroom_checkpoints

## 3. Результати

In [ ]:
# Перевірка збережених моделей
import os

checkpoint_dir = '/content/drive/MyDrive/mushroom_checkpoints'

if os.path.exists(checkpoint_dir):
    print("📁 Збережені файли:")
    for f in os.listdir(checkpoint_dir):
        size = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f"  - {f} ({size:.1f} MB)")
else:
    print("❌ Папка з чекпоінтами не знайдена")

In [ ]:
# Завантажити найкращу модель
from google.colab import files
import os

best_model = '/content/drive/MyDrive/mushroom_checkpoints/best_model.pth'

if os.path.exists(best_model):
    files.download(best_model)
    print("✅ Модель завантажено")
else:
    print("❌ Файл best_model.pth не знайдено. Тренування ще не завершено?")

## 4. Тестування моделі

In [ ]:
# Завантаження моделі та тест на одному зображенні
import torch
from PIL import Image
from torchvision import transforms
import sys
sys.path.insert(0, '/content/Applied_Mashroomatics')
from src.models.vit import create_vit_model

# Завантаження чекпоінта
checkpoint = torch.load('/content/drive/MyDrive/mushroom_checkpoints/best_model.pth')
num_classes = checkpoint['num_classes']
print(f"Кількість класів: {num_classes}")
print(f"Точність на валідації: {checkpoint['val_acc']:.2f}%")

# Створення моделі
model = create_vit_model(num_classes=num_classes, pretrained=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Модель завантажено і готова до використання")

In [ ]:
# Тест на зображенні
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

# Трансформації для inference
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Завантажте своє зображення
# image_path = '/content/drive/MyDrive/your_mushroom.jpg'
# image = Image.open(image_path).convert('RGB')
# input_tensor = transform(image).unsqueeze(0)

# with torch.no_grad():
#     output = model(input_tensor)
#     predicted_class = output.argmax(1).item()
#     confidence = torch.softmax(output, dim=1)[0][predicted_class].item()

# print(f"Predicted class: {predicted_class}")
# print(f"Confidence: {confidence:.2%}")

print("👆 Розкоментуйте код вище та вкажіть шлях до зображення гриба")

## ⚠️ Поради

**Якщо сесія падає:**
- Не закривайте комп'ютер / не давайте йому заснути
- Чекпоінти зберігаються на Drive - прогрес не втрачається

**Якщо CUDA out of memory:**
- Зменшіть `BATCH_SIZE` до 16
- Runtime → Restart runtime, потім запустіть знову

**Якщо GPU недоступний:**
- Runtime → Change runtime type → GPU (T4)
- Якщо T4 платний - зачекайте кілька годин або спробуйте Kaggle